In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path().resolve()
PROJECT_ROOT = NOTEBOOK_DIR.parent

RAW_ROOT = PROJECT_ROOT / "data" / "raw"
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed"

PHASE2B_DIR = PROCESSED_ROOT / "phase_02b_preprocess_imu"
PHASE2B_DIR.mkdir(parents=True, exist_ok=True)

print("NOTEBOOK_DIR :", NOTEBOOK_DIR)
print("PROJECT_ROOT :", PROJECT_ROOT)
print("RAW_ROOT     :", RAW_ROOT, "| exists:", RAW_ROOT.exists())
print("PHASE2B_DIR  :", PHASE2B_DIR)


In [ ]:
QC_ALL_PATH = PROJECT_ROOT / "data" / "processed" / "qc" / "00_QC_ALL_SUBJECTS.csv"
print("QC_ALL_PATH:", QC_ALL_PATH, "| exists:", QC_ALL_PATH.exists())

def load_qc_subject(qc_all_path: Path, subject_name: str) -> pd.DataFrame:
    qc_all = pd.read_csv(qc_all_path)
    qc_subj = qc_all[qc_all["subject"].astype(str) == str(subject_name)].copy()
    if qc_subj.empty:
        raise FileNotFoundError(f"No QC rows found for subject '{subject_name}' in {qc_all_path.name}")
    return qc_subj

def compute_trial_status(qc_subj: pd.DataFrame, gap_thr=0, sat_thr=0.01) -> pd.DataFrame:
    """
    Trial FAIL logic (same as Phase 2):
    - mapping issues
    - time gaps (IMU/EMG)
    - EMG issues only when EMG exists
    """
    qc = qc_subj.copy()
    qc["qc_fail"] = (
        (qc["mapping_ok"].fillna(True) == False) |
        (qc["emg_gaps"].fillna(0) > gap_thr) |
        (qc["imu_gyro_mag_gaps"].fillna(0) > gap_thr) |
        (qc["imu_acc_mag_gaps"].fillna(0) > gap_thr) |
        (
            (qc["emg_n_samples"].fillna(0) > 0) & (
                (qc["emg_lowvar_flag"].fillna(False) == True) |
                (qc["emg_saturation_ratio"].fillna(0) > sat_thr)
            )
        )
    )
    trial_status = qc.groupby("trial_id")["qc_fail"].any().reset_index()
    trial_status["status"] = np.where(trial_status["qc_fail"], "FAIL", "OK")
    return trial_status


In [ ]:
REPAIR_TRIALS = {
    
     "Healthy_Subject_5": {"lifting_exo"},
    # "ALS_Subject_10": {"drinking_noexo", "lifting_noexo"},
}


In [ ]:
import sys, inspect, importlib

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import trigno_io
importlib.reload(trigno_io)
from trigno_io import read_trigno_csv

print("read_trigno_csv imported from:", inspect.getfile(read_trigno_csv))


In [ ]:
from scipy.signal import butter, filtfilt

def butter_lowpass(cutoff_hz, fs, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, cutoff_hz/nyq, btype="lowpass")
    return b, a

def lowpass_filt(x, fs, cutoff_hz=10.0, order=4):
    x = np.asarray(x, dtype=float)
    x = np.nan_to_num(x, nan=0.0, posinf=0.0, neginf=0.0)
    b, a = butter_lowpass(cutoff_hz, fs, order=order)
    return filtfilt(b, a, x)



In [ ]:
def subject_csv_files(subject_name: str) -> list[Path]:
    subject_dir = RAW_ROOT / subject_name / "EMG&IMUTest"
    if not subject_dir.exists():
        raise FileNotFoundError(f"Subject dir not found: {subject_dir}")
    return sorted(subject_dir.glob("*.csv"))


In [ ]:
def preprocess_imu_trial(
    subject_name: str,
    csv_path: Path,
    qc_subj: pd.DataFrame,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    fs_fallback=148.148148,
):
    sensors, per_sensor, meta = read_trigno_csv(csv_path)
    trial_id = csv_path.stem

    rows = []
    skip_log = []

    for sensor in sensors:
        imu = per_sensor[sensor].get("imu", None)

        # skip sensors with no IMU (emg_only etc.)
        if imu is None or len(imu) == 0:
            skip_log.append({
                "subject": subject_name,
                "trial_id": trial_id,
                "sensor": sensor,
                "reason": "no_imu_in_reader_layout",
                "layout": meta.get("layout", ""),
                "block_kinds": meta.get("block_kinds", "")
            })
            continue

        # fs from QC (per trial/sensor), else fallback
        qrow = qc_subj[(qc_subj["trial_id"] == trial_id) & (qc_subj["sensor"] == sensor)]
        if len(qrow) == 1 and np.isfinite(qrow["imu_gyro_mag_fs_est"].values[0]):
            fs_imu = float(qrow["imu_gyro_mag_fs_est"].values[0])
        else:
            fs_imu = fs_fallback

        t = imu["t_imu"].to_numpy()

        gx = imu["gyro_x_dps"].to_numpy()
        gy = imu["gyro_y_dps"].to_numpy()
        gz = imu["gyro_z_dps"].to_numpy()

        gyro_mag = np.sqrt(gx*gx + gy*gy + gz*gz)
        gyro_gate = lowpass_filt(gyro_mag, fs=fs_imu, cutoff_hz=lp_cutoff, order=4)

        out = pd.DataFrame({
            "subject": subject_name,
            "trial_id": trial_id,
            "sensor": sensor,
            "t_imu": t,

            # keep axes (needed later)
            "gyro_x_dps": gx,
            "gyro_y_dps": gy,
            "gyro_z_dps": gz,
            "gyro_mag": gyro_mag,
            "gyro_gate": gyro_gate,

            "fs_imu_used": fs_imu,
        })

        if compute_acc_dyn:
            ax = imu["acc_x_g"].to_numpy()
            ay = imu["acc_y_g"].to_numpy()
            az = imu["acc_z_g"].to_numpy()

            acc_mag = np.sqrt(ax*ax + ay*ay + az*az)

            # baseline removal (very low LP) -> dynamic component
            acc_baseline = lowpass_filt(acc_mag, fs=fs_imu, cutoff_hz=0.5, order=2)
            acc_dyn = acc_mag - acc_baseline
            acc_gate = lowpass_filt(np.abs(acc_dyn), fs=fs_imu, cutoff_hz=lp_cutoff, order=4)

            out["acc_x_g"] = ax
            out["acc_y_g"] = ay
            out["acc_z_g"] = az
            out["acc_mag"] = acc_mag
            out["acc_dyn"] = acc_dyn
            out["acc_gate"] = acc_gate

        rows.append(out)

    df = pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()
    skip_df = pd.DataFrame(skip_log)
    return df, skip_df


In [ ]:
def run_phase2b_for_subject(
    subject_name: str,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
    fs_fallback=148.148148,
):
    files = subject_csv_files(subject_name)
    print("Files found:", len(files))

    qc_subj = load_qc_subject(QC_ALL_PATH, subject_name)
    trial_status = compute_trial_status(qc_subj, gap_thr=gap_thr, sat_thr=sat_thr)

    ok_trials = set(trial_status.loc[trial_status["status"] == "OK", "trial_id"].astype(str))
    repair_trials = REPAIR_TRIALS.get(subject_name, set())
    allowed_trials = ok_trials | set(repair_trials)

    print(f"OK trials: {len(ok_trials)} | FAIL trials: {int((trial_status['status']=='FAIL').sum())}")
    if repair_trials:
        print("Repair-enabled trials:", sorted(repair_trials))

    files_allowed = [fp for fp in files if fp.stem in allowed_trials]
    print("Files to preprocess (OK + repair):", len(files_allowed))
    print("Allowed stems:", [fp.stem for fp in files_allowed])

    all_df = []
    all_skip = []

    for fp in files_allowed:
        df_trial, skip_df = preprocess_imu_trial(
            subject_name=subject_name,
            csv_path=fp,
            qc_subj=qc_subj,
            lp_cutoff=lp_cutoff,
            compute_acc_dyn=compute_acc_dyn,
            fs_fallback=fs_fallback,
        )
        if not df_trial.empty:
            all_df.append(df_trial)
        if not skip_df.empty:
            all_skip.append(skip_df)

    subj_imu = pd.concat(all_df, ignore_index=True) if all_df else pd.DataFrame()
    skip_all = pd.concat(all_skip, ignore_index=True) if all_skip else pd.DataFrame()

    # save ONLY one file
    out_path = PHASE2B_DIR / f"{subject_name}__imu_gate.parquet"
    if subj_imu.empty:
        print("WARNING: No IMU rows produced. Nothing saved.")
    else:
        subj_imu.to_parquet(out_path, index=False)
        print("Saved:", out_path)

    # small summaries for display (not saved)
    if not skip_all.empty:
        skip_summary = (
            skip_all.groupby("reason")["sensor"]
            .count()
            .reset_index()
            .rename(columns={"sensor": "n_skipped"})
            .sort_values("n_skipped", ascending=False)
        )
    else:
        skip_summary = pd.DataFrame(columns=["reason", "n_skipped"])

    return subj_imu, trial_status, skip_summary



In [ ]:
SUBJECT_NAME = "ALS_Subject_1"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())



In [ ]:
import matplotlib.pyplot as plt

TRIAL_ID = "drinking_exo"
SENSOR = "BicBrachii"
T_START, T_END = 3.0, 8.0

seg = subj_imu[
    (subj_imu["trial_id"] == TRIAL_ID) &
    (subj_imu["sensor"] == SENSOR) &
    (subj_imu["t_imu"] >= T_START) & (subj_imu["t_imu"] <= T_END)
].copy()

plt.figure()
plt.plot(seg["t_imu"], seg["gyro_mag"])
plt.title(f"gyro_mag | {SUBJECT_NAME} | {TRIAL_ID} | {SENSOR}")
plt.xlabel("Time (s)")
plt.ylabel("deg/s (magnitude)")
plt.show()

plt.figure()
plt.plot(seg["t_imu"], seg["gyro_gate"])
plt.title(f"gyro_gate (LP) | {SUBJECT_NAME} | {TRIAL_ID} | {SENSOR}")
plt.xlabel("Time (s)")
plt.ylabel("LP filtered gyro magnitude")
plt.show()

if "acc_gate" in seg.columns:
    plt.figure()
    plt.plot(seg["t_imu"], seg["acc_gate"])
    plt.title(f"acc_gate (LP) | {SUBJECT_NAME} | {TRIAL_ID} | {SENSOR}")
    plt.xlabel("Time (s)")
    plt.ylabel("LP filtered |acc_dyn|")
    plt.show()


subject 2

In [ ]:
SUBJECT_NAME = "ALS_Subject_2"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 3

In [ ]:
SUBJECT_NAME = "ALS_Subject_3"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 4

In [ ]:
SUBJECT_NAME = "ALS_Subject_4"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 5

In [ ]:
SUBJECT_NAME = "ALS_Subject_5"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 7

In [ ]:
SUBJECT_NAME = "ALS_Subject_7"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 8

In [ ]:
SUBJECT_NAME = "ALS_Subject_8"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 9

In [ ]:
SUBJECT_NAME = "ALS_Subject_9"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 10

In [ ]:
SUBJECT_NAME = "ALS_Subject_10"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 11

In [ ]:
SUBJECT_NAME = "ALS_Subject_11"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 12

In [ ]:
SUBJECT_NAME = "ALS_Subject_12"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 13

In [ ]:
SUBJECT_NAME = "ALS_Subject_13"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 14

In [ ]:
SUBJECT_NAME = "ALS_Subject_14"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 15

In [ ]:
SUBJECT_NAME = "ALS_Subject_15"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

subject 16

In [ ]:
SUBJECT_NAME = "ALS_Subject_16"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 1

In [ ]:
SUBJECT_NAME = "Healthy_Subject_1"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 2

In [ ]:
SUBJECT_NAME = "Healthy_Subject_2"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 3

In [ ]:
SUBJECT_NAME = "Healthy_Subject_3"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 4

In [ ]:
SUBJECT_NAME = "Healthy_Subject_4"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 5 

In [ ]:
SUBJECT_NAME = "Healthy_Subject_5"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 6

In [ ]:
SUBJECT_NAME = "Healthy_Subject_6"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 7

In [ ]:
SUBJECT_NAME = "Healthy_Subject_7"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())

healthy 8

In [ ]:
SUBJECT_NAME = "Healthy_Subject_8"   # change only this each run

subj_imu, trial_status, skip_summary = run_phase2b_for_subject(
    SUBJECT_NAME,
    lp_cutoff=10.0,
    compute_acc_dyn=True,
    gap_thr=0,
    sat_thr=0.01,
)

display(trial_status.sort_values("trial_id"))
display(skip_summary)

display(
    subj_imu.groupby(["trial_id"])["gyro_gate"].agg(["min", "median", "max"]).reset_index()
)
display(subj_imu.head())